# 01: Reproduce Experiments

This notebook runs the full experimental pipeline for a single model profile.
It is designed for both local (RTX 4090) and cloud (A100/H100) execution.

**Phases:**
1. **Compliance test** — Quick 2000-token test across all prompt conditions
2. **Phase 1: Extended baseline** — N full-length runs with activation capture
3. **Phase 2: Truncation analysis** — CPU-only post-processing of Phase 1
4. **Phase 3: Control runs** — All conditions with loop-detection early termination
5. **Phase 4: Direction extraction** — Contrastive directions + topic vector comparison

**To reproduce on cloud (Llama 70B):**
1. Set `PROFILE = "cloud"` in the configuration cell
2. Ensure HuggingFace token with Llama access is configured
3. Run all cells

Results are saved to `outputs/runs/{profile}_replication/` in the format
expected by `02_generate_figures.ipynb`.

## Configuration

Set the profile and run parameters here. Everything else is automated.

In [ ]:
# ========================================================================
# CONFIGURATION — Edit this cell for your environment
# ========================================================================

# Profile: "local" (Llama 8B), "mistral", "gemma", "qwen", "cloud" (Llama 70B)
PROFILE = "cloud"

# Override run counts (None = use profile defaults)
N_BASELINE_RUNS = None      # Phase 1: extended baseline (default: 50 for cloud)
N_CONTROL_RUNS = None       # Phase 3: runs per condition (default: 10 for cloud)

# Which phases to run (set False to skip)
RUN_COMPLIANCE = True       # Quick compliance test (~10 min)
RUN_PHASE1 = True           # Extended baseline (~2-6 hours depending on model)
RUN_PHASE2 = True           # Truncation analysis (CPU, ~5 min)
RUN_PHASE3 = True           # Control runs (~1-4 hours)
RUN_PHASE4 = True           # Direction extraction (~15 min)

# Environment
import os
os.environ.setdefault("HF_HOME", "/storage/huggingface")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"

In [ ]:
import sys, json, time, gc
from pathlib import Path
from datetime import datetime

# Add project root to path
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))

import torch
import numpy as np

from scripts.llama_replication import (
    PROFILES, PROMPT_CONDITIONS, GENERATION_PARAMS,
    load_model,
    phase1_extended_baseline,
    phase2_truncation_analysis,
    phase3_controls,
    phase4_directions,
)
from src.analysis.loop_detection import parse_observations

# Resolve profile and output directory
profile = PROFILES[PROFILE]
output_dir = PROJECT / "outputs" / "runs" / f"{PROFILE}_replication"
output_dir.mkdir(parents=True, exist_ok=True)

n_baseline = N_BASELINE_RUNS or profile["n_runs_default"]
n_control = N_CONTROL_RUNS or profile["n_control_runs_default"]

print(f"Profile:          {PROFILE} ({profile['model_id']})")
print(f"Output:           {output_dir}")
print(f"Baseline runs:    {n_baseline}")
print(f"Control runs:     {n_control} per condition")
print(f"Conditions:       {len(PROMPT_CONDITIONS)}")
print(f"Capture layers:   {profile['capture_layers']}")
print(f"Hotspot layer:    {profile['hotspot_layer']}")
print(f"GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM:             {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# Save config
config = {
    "profile": PROFILE,
    "model_id": profile["model_id"],
    "n_baseline": n_baseline,
    "n_control": n_control,
    "conditions": list(PROMPT_CONDITIONS.keys()),
    "capture_layers": profile["capture_layers"],
    "generation_params": GENERATION_PARAMS,
    "timestamp": datetime.now().isoformat(),
}
(output_dir / "config.json").write_text(json.dumps(config, indent=2))

## Compliance Test

Quick screen: does the model follow the Pull Methodology format for each prompt condition?
Generates 2000 tokens per condition and counts parsed observations.

This produces the data for the cross-model compliance heatmap (Figure 11).

In [ ]:
if RUN_COMPLIANCE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    print(f"Loading {profile['model_id']} for compliance test...")
    tokenizer = AutoTokenizer.from_pretrained(profile["model_id"], trust_remote_code=True)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        profile["model_id"], quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )
    model.eval()
    device = next(model.parameters()).device

    compliance = {}
    for name, prompt in PROMPT_CONDITIONS.items():
        messages = [{"role": "user", "content": prompt}]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text_input, return_tensors="pt").to(device)
        prompt_length = inputs["input_ids"].shape[1]

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=2000, temperature=0.7,
                top_p=1.0, do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.time() - t0

        generated_ids = outputs[0][prompt_length:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        n_tokens = len(generated_ids)
        observations = parse_observations(text)

        status = "COMPLIANT" if len(observations) >= 10 else ("PARTIAL" if len(observations) >= 1 else "REFUSED")
        compliance[name] = {
            "n_tokens": n_tokens, "n_observations": len(observations),
            "elapsed_s": round(elapsed, 1), "tok_s": round(n_tokens / elapsed, 1),
            "status": status, "first_500_chars": text[:500],
        }
        print(f"  {name:<25} {status:<10} obs={len(observations):>4}  tok={n_tokens:>5}")

        del outputs
        torch.cuda.empty_cache()

    out_path = output_dir / "compliance_test.json"
    out_path.write_text(json.dumps(compliance, indent=2))
    print(f"\nSaved to {out_path}")

    # Free memory before pipeline phases
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Compliance test skipped.")

## Phase 1: Extended Baseline Runs

Full-length generations with Dadfar's exact prompt and parameters.
No early termination — captures the complete trajectory including limit cycle.

Checkpoints after each run, so this cell is resumable.

In [ ]:
model = tokenizer = None

if RUN_PHASE1 or RUN_PHASE3 or RUN_PHASE4:
    model, tokenizer = load_model(profile)

if RUN_PHASE1:
    print(f"\n{'='*60}")
    print(f"PHASE 1: Extended Baseline ({n_baseline} runs)")
    print(f"{'='*60}")
    phase1_extended_baseline(model, tokenizer, profile, output_dir, n_baseline)
else:
    print("Phase 1 skipped.")

## Phase 2: Truncation Analysis (CPU)

Post-processes Phase 1 data to measure metric stability at various observation cutoffs.
Demonstrates that spectral_power_low never converges (length confound) while
other metrics stabilize by observation ~100.

In [ ]:
if RUN_PHASE2:
    print(f"\n{'='*60}")
    print(f"PHASE 2: Truncation Analysis")
    print(f"{'='*60}")
    phase2_truncation_analysis(output_dir)
else:
    print("Phase 2 skipped.")

## Phase 3: Control Runs with Early Termination

Runs all prompt conditions (including refusal-study conditions) with
loop-detection early termination. This is the main controlled experiment:
same task structure, different content.

Checkpoints after each run.

In [ ]:
if RUN_PHASE3:
    print(f"\n{'='*60}")
    print(f"PHASE 3: Control Runs ({len(PROMPT_CONDITIONS)} conditions x {n_control} runs)")
    print(f"{'='*60}")
    phase3_controls(model, tokenizer, profile, output_dir, n_control)
else:
    print("Phase 3 skipped.")

## Phase 4: Direction Extraction

Extracts Dadfar's contrastive direction (self-referential minus non-self-referential)
and a control topic direction (forest minus math). Tests both on held-out prompts
to show they are structurally equivalent topic vectors.

In [ ]:
if RUN_PHASE4:
    print(f"\n{'='*60}")
    print(f"PHASE 4: Direction Extraction")
    print(f"{'='*60}")
    phase4_directions(model, tokenizer, profile, output_dir)
else:
    print("Phase 4 skipped.")

# Cleanup
if model is not None:
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("\nModel unloaded.")

## Summary

Lists all output files generated by this run.

In [ ]:
print(f"\n{'='*60}")
print(f"EXPERIMENT COMPLETE: {PROFILE}")
print(f"{'='*60}")
print(f"Output directory: {output_dir}\n")

for p in sorted(output_dir.rglob("*.json")):
    size_kb = p.stat().st_size / 1024
    rel = p.relative_to(output_dir)
    print(f"  {str(rel):55s} {size_kb:8.1f} KB")

n_txt = len(list(output_dir.rglob("*.txt")))
n_npy = len(list(output_dir.rglob("*.npy")))
print(f"\n  + {n_txt} text files, {n_npy} activation vectors")
print(f"\nNext: run 02_generate_figures.ipynb to produce publication figures.")